In [14]:
!pip install -q langchain-groq
%pip install -q langchain-classic

In [25]:
from langchain_groq import ChatGroq
from pathlib import Path
import os
from langchain_core.output_parsers import StrOutputParser, CommaSeparatedListOutputParser

In [10]:
def _read_env_file(name, filename="secrets.env"):
    """Look for KEY=value in secrets.env in the working dir or /content."""
    for folder in (Path.cwd(), Path("/content")):
        path = folder / filename
        if path.is_file():
            for line in path.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, _, val = line.partition("=")
                if key.strip().removeprefix("export ").strip() == name:
                    return val.strip().strip('"').strip("'")
    return None

def get_secret(name, required=True):
    value = None
    try:
        value = userdata.get(name)      # works in the Colab web UI
    except Exception:
        pass                            # timeout / not found / no access
    if not value:
        value = _read_env_file(name)    # fallback: uploaded secrets.env
    if value:
        os.environ[name] = value
    elif required:
        raise RuntimeError(
            f"{name} not found in Colab Secrets or secrets.env. "
            "Upload secrets.env to the Colab server and re-run."
        )
    return value

groq_key = get_secret("GROQ_API_KEY")


print("GROQ key loaded:", bool(groq_key))


GROQ key loaded: True


In [11]:
llm = ChatGroq(
        model = "openai/gpt-oss-120b",
        model_kwargs= {"seed" : 350 },
        temperature= 0,
        max_tokens= 250
)

**String output Parser**

In [21]:
template_h = ('human', 'Hey, who is Nvidia CEO ?')

response = llm.invoke([template_h])

response

AIMessage(content='Nvidia’s chief executive officer is **Jensen\u202fHuang**. He co‑founded the company in 1993 and has served as its CEO ever since, guiding Nvidia from a graphics‑chip maker into a leader in AI hardware, data‑center accelerators, and autonomous‑vehicle technology.', additional_kwargs={'reasoning_content': 'The user asks: "Hey, who is Nvidia CEO ?" This is a straightforward factual question. The answer: Jensen Huang. Provide current info. As of 2026, Jensen Huang remains CEO. Provide brief info.'}, response_metadata={'token_usage': {'completion_tokens': 116, 'prompt_tokens': 78, 'total_tokens': 194, 'completion_time': 0.24101648, 'completion_tokens_details': {'reasoning_tokens': 45}, 'prompt_time': 0.004258347, 'prompt_tokens_details': None, 'queue_time': 0.146861422, 'total_time': 0.245274827}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_dee443f41b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, i

In [22]:
string_parser = StrOutputParser()

parsed_data = string_parser.invoke(response)

parsed_data

'Nvidia’s chief executive officer is **Jensen\u202fHuang**. He co‑founded the company in 1993 and has served as its CEO ever since, guiding Nvidia from a graphics‑chip maker into a leader in AI hardware, data‑center accelerators, and autonomous‑vehicle technology.'

**List Output Parser**

In [35]:
list_sep = CommaSeparatedListOutputParser().get_format_instructions()

print(list_sep)

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [36]:
template_h = (
    'human',  
    '''can you list down some exciting projects of nvidia ?  
       {list_sep}
    '''
    )

response = llm.invoke([template_h])

response

AIMessage(content='NVIDIA RTX Real-Time Ray Tracing{list_sep}DLSS (Deep Learning Super Sampling){list_sep}NVIDIA Omniverse Platform{list_sep}CUDA Parallel Computing Platform{list_sep}NVIDIA DGX Systems for AI Research{list', additional_kwargs={'reasoning_content': 'The user asks: "can you list down some exciting projects of nvidia ? {list_sep}". They want a list of exciting projects of Nvidia. Provide a list separated by the placeholder {list_sep}. The user includes {list_sep} token, which likely indicates they want a specific separator. We need to output a list with that token as separator between items. Probably each item separated by that token. So we need to produce something like "Project A{list_sep}Project B{list_sep}Project C". Provide a list of Nvidia projects: e.g., RTX GPUs, DLSS, Omniverse, CUDA, DGX, Jetson, Clara, DRIVE, AI research like Megatron, etc. Provide maybe 8-10 items. Use the token exactly as given. No extra formatting. Just the list. Ensure no extra spaces? Prob

In [37]:
Parsed_list = CommaSeparatedListOutputParser()

parsed_list_response = Parsed_list.invoke(response)

parsed_list_response

['NVIDIA RTX Real-Time Ray Tracing{list_sep}DLSS (Deep Learning Super Sampling){list_sep}NVIDIA Omniverse Platform{list_sep}CUDA Parallel Computing Platform{list_sep}NVIDIA DGX Systems for AI Research{list']